In [24]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [25]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import json

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque

In [26]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 75  # 75 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 30  # 30 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [27]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [28]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [29]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [30]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [31]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)

        return self

    def clean_data(self):
        # Drop volume if zero or NaN
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        # RSI
        self.df.sort_index(inplace=True)
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_swing_points(self, left_bars=10, right_bars=10):
        # Identify local swing highs/lows
        self.df.sort_index(inplace=True)

        rolling_high = self.df['high'].rolling(left_bars + 1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars + 1, center=False).min()

        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars + 1, center=False).max()
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars + 1, center=False).min()

        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        # Donchian Channels
        self.df.sort_index(inplace=True)

        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Donchian breakout flags
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion measure
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        self.df.sort_index(inplace=True)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        # Historical volatility, Z-score
        self.df.sort_index(inplace=True)

        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)

        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        # Volume-based features if present
        if 'volume' not in self.df.columns:
            return self

        self.df.sort_index(inplace=True)

        # OBV
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spikes
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP approximation
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        # Classify trending vs ranging via ADX threshold
        self.df.sort_index(inplace=True)

        if 'adx' not in self.df.columns:
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        return self

    def add_time_features(self):
        # Hour, day-of-week, cyclical encodings
        self.df.sort_index(inplace=True)

        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        return self

    def add_adaptive_targets_and_stops(self):
        # Adapt targets/stops based on regime
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def label_signals(self):
        """
        Priority is profit first (buy_target/sell_target),
        then stop-loss (buy_sl/sell_sl).
        """
        self.df['Signal'] = 0
        self.df['Entry Price'] = 0.0
        self.df['Exit Price'] = 0.0
        self.df['candles_to_profit'] = 0
        self.df['candles_to_loss'] = 0

        all_idx = self.df.index.tolist()
        n = len(all_idx)

        for i in range(n - 1):
            idx_i = all_idx[i]

            entry_price = self.df.at[idx_i, 'close']
            target = self.df.at[idx_i, 'Target']
            stop_loss = self.df.at[idx_i, 'StopLoss']

            # Hypothetical buy/sell levels
            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss
            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            future_slice = all_idx[i+1:]
            signal_found = False

            for offset, future_idx in enumerate(future_slice, start=1):
                future_high = self.df.at[future_idx, 'high']
                future_low = self.df.at[future_idx, 'low']

                triggers = []
                # Check buy/sell target/SL
                if future_high >= buy_target_price:
                    triggers.append(('buy_target', offset))
                if future_low <= buy_sl_price:
                    triggers.append(('buy_sl', offset))
                if future_low <= sell_target_price:
                    triggers.append(('sell_target', offset))
                if future_high >= sell_sl_price:
                    triggers.append(('sell_sl', offset))

                # If multiple triggers this candle, pick the earliest by priority
                if triggers:
                    priority_map = {
                        'buy_target': 1,
                        'sell_target': 2,
                        'buy_sl': 3,
                        'sell_sl': 4
                    }
                    triggers.sort(key=lambda x: priority_map[x[0]])
                    first_trigger, candle_count = triggers[0]

                    if first_trigger == 'buy_target':
                        self.df.at[idx_i, 'Signal'] = 1
                        self.df.at[idx_i, 'Exit Price'] = buy_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'sell_target':
                        self.df.at[idx_i, 'Signal'] = 3
                        self.df.at[idx_i, 'Exit Price'] = sell_target_price
                        self.df.at[idx_i, 'candles_to_profit'] = candle_count

                    elif first_trigger == 'buy_sl':
                        self.df.at[idx_i, 'Signal'] = 2
                        self.df.at[idx_i, 'Exit Price'] = buy_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    elif first_trigger == 'sell_sl':
                        self.df.at[idx_i, 'Signal'] = 4
                        self.df.at[idx_i, 'Exit Price'] = sell_sl_price
                        self.df.at[idx_i, 'candles_to_loss'] = candle_count

                    self.df.at[idx_i, 'Entry Price'] = entry_price
                    signal_found = True
                    break

            # If no trigger in future
            if not signal_found:
                self.df.at[idx_i, 'Signal'] = 0
                self.df.at[idx_i, 'Entry Price'] = entry_price

        return self

    def run_pipeline(self):
        (
            self.preprocess_datetime()
                .clean_data()
                .add_base_timeframe_features()
                .add_swing_points()
                .add_range_breakout_features()
                .add_momentum_features()
                .add_volatility_features()
                .add_volume_features()
                .add_regime_features()
                .add_time_features()
                .add_adaptive_targets_and_stops()
                .label_signals()
        )
        return self

    def get_processed_df(self):
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [ ]:
nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

# Fetch candle data for each instrument and each timeframe
nifty_2m = fetch_train_candle_data(3, nf_index_symbol, 2)
nifty_5m = fetch_train_candle_data(3, nf_index_symbol, 5)
nifty_15m = fetch_train_candle_data(3, nf_index_symbol, 15)

bnf_2m = fetch_train_candle_data(3, bnf_index_symbol, 2)
bnf_5m = fetch_train_candle_data(3, bnf_index_symbol, 5)
bnf_15m = fetch_train_candle_data(3, bnf_index_symbol, 15)

print("Fetched historical data for all instruments")

# Clean and remove duplicate datetimes (if any)
nifty_2m = nifty_2m.drop_duplicates(subset='datetime', keep='first')
nifty_5m = nifty_5m.drop_duplicates(subset='datetime', keep='first')
nifty_15m = nifty_15m.drop_duplicates(subset='datetime', keep='first')

bnf_2m = bnf_2m.drop_duplicates(subset='datetime', keep='first')
bnf_5m = bnf_5m.drop_duplicates(subset='datetime', keep='first')
bnf_15m = bnf_15m.drop_duplicates(subset='datetime', keep='first')

# Nifty after processing
nf_train_pipeline_2m = FullFeaturePipeline(nifty_2m)
nf_train_pipeline_2m.run_pipeline()
df_nifty_2m = nf_train_pipeline_2m.get_processed_df()

nf_train_pipeline_5m = FullFeaturePipeline(nifty_5m)
nf_train_pipeline_5m.run_pipeline()
df_nifty_5m = nf_train_pipeline_5m.get_processed_df()

nf_train_pipeline_15m = FullFeaturePipeline(nifty_15m)
nf_train_pipeline_15m.run_pipeline()
df_nifty_15m = nf_train_pipeline_15m.get_processed_df()

# Bank Nifty after processing
bnf_train_pipeline_2m = FullFeaturePipeline(bnf_2m)
bnf_train_pipeline_2m.run_pipeline()
df_bnf_2m = bnf_train_pipeline_2m.get_processed_df()

bnf_train_pipeline_5m = FullFeaturePipeline(bnf_5m)
bnf_train_pipeline_5m.run_pipeline()
df_bnf_5m = bnf_train_pipeline_5m.get_processed_df()

bnf_train_pipeline_15m = FullFeaturePipeline(bnf_15m)
bnf_train_pipeline_15m.run_pipeline()
df_bnf_15m = bnf_train_pipeline_15m.get_processed_df()

In [ ]:
df_bnf_2m.columns

In [ ]:
df_bnf_2m

In [ ]:
import gym
import numpy as np
import pandas as pd
from gym import spaces

class Position:
    """Tracks ongoing position state with SL/Target management."""
    def __init__(self):
        self.status = "flat"
        self.entry_price = None
        self.entry_time = None
        self.sl_points = None
        self.target_points = None
        self.direction = 0  # +1 long, -1 short
        self.quantity = 0
        self.time_in_trade = 0

    def reset(self):
        self.__init__()

    def update_unrealized_pnl(self, current_price: float) -> float:
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

class TradingEnv(gym.Env):
    """Final trading environment, with config-based parameters."""
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        # env_config is a dict with keys: "df", "lot_size", "transaction_cost", "window_size"
        self.window_size = env_config["window_size"]
        self.df = env_config["df"].drop(columns=['Signal', 'candles_to_profit', 'candles_to_loss']).copy()
        self.lot_size = env_config["lot_size"]
        self.transaction_cost = env_config["transaction_cost"]
        self.instrument_name = env_config.get("name", "Unknown")

        # We scale initial capital so there's enough buffer for trades
        self.initial_capital = self.df['high'].max() * self.lot_size * 3
        self.current_capital = None
        self.current_step = None
        self.position = Position()
        self.trade_log = []

        # Define action space: 0=Hold, 1=Buy, 2=Sell, 3=Close
        self.action_space = spaces.Discrete(4)

        # Define observation space
        n_features = len(self.df.columns)
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32)
        })

    def reset(self):
        self.current_step = self.window_size
        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log = []
        return self._get_observation()

    def step(self, action: int):
        current_data = self.df.iloc[self.current_step]
        done = False
        reward = 0.0

        # Check if SL is hit
        if self._check_sl_hit(current_data):
            reward += self._close_position(current_data, "SL Hit")

        # Execute action
        self._execute_action(action, current_data)

        # Time in trade increments
        if self.position.status != "flat":
            self.position.time_in_trade += 1

        # Calculate unrealized PnL
        unrealized_pnl = 0.0
        if self.position.status != "flat":
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
        reward += self._calculate_reward(unrealized_pnl)

        # Increment step
        self.current_step += 1
        if self.current_step >= len(self.df) or self.current_capital <= 0:
            done = True
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")

        return self._get_observation(), reward, done, {}

    def _get_observation(self) -> dict:
        market_features = self.df.iloc[self.current_step].values.astype(np.float32)
        # Position context
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            (self.position.update_unrealized_pnl(self.df.iloc[self.current_step]['close'])
             if self.position.status != "flat" else 0.0) / self.initial_capital,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        # History of market features (window_size steps)
        history_start = max(0, self.current_step - self.window_size)
        history = self.df.iloc[history_start:self.current_step].values.astype(np.float32)
        if len(history) < self.window_size:
            padding = np.zeros((self.window_size - len(history), self.df.shape[1]))
            history = np.vstack([padding, history])

        return {"market": market_features, "position": position_context, "history": history}

    def _execute_action(self, action: int, current_data: pd.Series):
        price = current_data['open']
        if action == 3 and self.position.status != "flat":
            # Manual close
            self._close_position(current_data, "Manual Close")
        elif action == 1:
            # Buy
            if self.position.status == "short":
                self._close_position(current_data, "Reverse Close")
            if self.position.status == "flat" and self.current_capital >= price * self.lot_size:
                self._open_position("long", current_data)
        elif action == 2:
            # Sell
            if self.position.status == "long":
                self._close_position(current_data, "Reverse Close")
            if self.position.status == "flat" and self.current_capital >= price * self.lot_size:
                self._open_position("short", current_data)

    def _open_position(self, direction: str, current_data: pd.Series):
        entry_price = current_data['open']
        self.current_capital -= entry_price * self.lot_size   # Deduct cost (placeholder logic)
        self.position.open(
            entry_price=entry_price,
            sl_points=current_data['StopLoss'],
            target_points=current_data['Target'],
            direction=1 if direction == "long" else -1,
            quantity=self.lot_size
        )
        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None
        })

    def _close_position(self, current_data: pd.Series, reason: str) -> float:
        if self.position.status == "flat":
            return 0.0
        if "SL" in reason:
            # Calculate SL price based on direction
            if self.position.direction == 1:
                exit_price = self.position.entry_price - self.position.sl_points
            else:
                exit_price = self.position.entry_price + self.position.sl_points
    else:
        exit_price = current_data['open']
        realized_pnl = ((exit_price - self.position.entry_price)
                        * self.position.direction * self.lot_size)
        # Subtract transaction costs
        realized_pnl -= self.transaction_cost
        self.current_capital += realized_pnl
        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason
                })
                break
        self.position.reset()
        return realized_pnl / self.initial_capital

    def _check_sl_hit(self, current_data: pd.Series) -> bool:
        if self.position.status == "flat":
            return False
        # For a "long", SL is entry_price - sl_points
        # For a "short", SL is entry_price + sl_points
        if self.position.direction == 1:
            sl_price = self.position.entry_price - self.position.sl_points
            return current_data['low'] <= sl_price
        else:
            sl_price = self.position.entry_price + self.position.sl_points
            return current_data['high'] >= sl_price

    def _calculate_reward(self, unrealized_pnl: float) -> float:
        # Simple shaping: 0.1 * scaled PnL - penalty for time in trade, etc.
        reward = 0.1 * (unrealized_pnl / self.initial_capital)
        reward -= 0.02 * (self.position.time_in_trade / 100)
        current_value = self.current_capital + unrealized_pnl
        peak_value = max(self.initial_capital, current_value)
        drawdown = (peak_value - current_value) / peak_value
        if drawdown > 0.05:
            reward -= 0.5 * drawdown
        return float(np.clip(reward, -1.0, 1.0))

    def get_metrics(self) -> dict:
        if not self.trade_log:
            return {}
        winning_trades = [t for t in self.trade_log if t['pnl'] is not None and t['pnl'] > 0]
        losing_trades = [t for t in self.trade_log if t['pnl'] is not None and t['pnl'] <= 0]
        return {
            "instrument": self.instrument_name,
            "initial_capital": self.initial_capital,
            "final_capital": self.current_capital,
            "total_trades": len(self.trade_log),
            "win_rate": len(winning_trades) / len(self.trade_log) if self.trade_log else 0,
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": (sum(t['pnl'] for t in winning_trades) /
                              abs(sum(t['pnl'] for t in losing_trades))
                              if losing_trades else np.inf),
        }

    def _compute_max_drawdown(self) -> float:
        equity_curve = [self.initial_capital]
        for trade in self.trade_log:
            if trade['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + trade['pnl'])
        equity_curve = np.array(equity_curve)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return dd.max()

In [ ]:
import torch

config = {
    "instruments": [
        {
            "name": "NIFTY_15M",
            "df": df_nifty_15m,            # Preprocessed dataframe
            "lot_size": nf_quantity,              # Example lot size
            "transaction_cost": 20.0      # Example total brokerage/slippage
        },
        {
            "name": "BANKNIFTY_15M",
            "df": df_bnf_15m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_5M",
            "df": df_nifty_5m,
            "lot_size": nf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_5M",
            "df": df_bnf_5m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "NIFTY_2M",
            "df": df_nifty_2m,
            "lot_size": nf_quantity,
            "transaction_cost": 20.0
        },
        {
            "name": "BANKNIFTY_2M",
            "df": df_bnf_2m,
            "lot_size": bnf_quantity,
            "transaction_cost": 20.0
        },
    ],
    "window_size": 30,              # Observation window
    "meta_epochs": 10,             # Number of meta-training epochs
    "inner_epochs": 2,             # Number of inner steps/epochs per task
    "gamma": 0.99,                 # Discount factor
    "lr_meta": 1e-3,               # Meta learning rate
    "lr_inner": 1e-2,              # Inner loop learning rate
    "batch_size": 512,             # Batch size for sampling from each env
    "hidden_size": 128,            # Hidden size for policy/value networks
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")               # "cuda" if GPU available, else "cpu"
}

In [ ]:

#!/usr/bin/env python
import gym
import numpy as np
import pandas as pd
import torch
from gym import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# ------------------ Training and Evaluation ------------------

if __name__ == '__main__':
    # For efficient learning, provide higher timeframe data first (e.g., 15M) and then lower timeframes.
    # Here we iterate through each instrument in our config.
    for instrument in config["instruments"]:
        print(f"\n=== Training PPO for instrument: {instrument['name']} ===")
        env_config = {
            "df": instrument["df"],
            "lot_size": instrument["lot_size"],
            "transaction_cost": instrument["transaction_cost"],
            "window_size": config["window_size"],
            "name": instrument["name"]
        }
        env = TradingEnv(env_config)
        check_env(env, warn=True)

        model = PPO("MultiInputPolicy", env, verbose=1, device=config["device"])
        model.learn(total_timesteps=10000)  # Adjust total_timesteps as needed

        # Run one test episode using the trained model.
        obs = env.reset()
        done = False
        while not done:
            action, _states = model.predict(obs)
            obs, reward, done, info = env.step(action)

        metrics = env.get_metrics()
        print("\nTrade Log Details:")
        for trade in env.trade_log:
            if trade["exit_time"] is not None:
                points = (trade["exit_price"] - trade["entry_price"]) if trade["position"] == "long" else (trade["entry_price"] - trade["exit_price"])
                print(f"Entry: {trade['entry_time']}, Exit: {trade['exit_time']}, Position: {trade['position']}, "
                      f"Entry Price: {trade['entry_price']:.2f}, Exit Price: {trade['exit_price']:.2f}, "
                      f"Points: {points:.2f}, PnL: {trade['pnl']:.2f}, Reason: {trade['reason']}")
            else:
                print(f"Open trade at {trade['entry_time']}")

        print("\nSummary Metrics:")
        print(f"Initial Capital: {metrics.get('initial_capital'):.2f}")
        print(f"Final Capital: {metrics.get('final_capital'):.2f}")
        print(f"Total Trades: {metrics.get('total_trades')}")
        print(f"Win Rate: {metrics.get('win_rate'):.2f}%")
        print(f"Max Drawdown: {metrics.get('max_drawdown'):.2f}")
        print(f"Profit Factor: {metrics.get('profit_factor'):.2f}")

In [ ]:
#!/usr/bin/env python
import gym
import numpy as np
import pandas as pd
import torch
from gym import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

# ------------------ Training and Evaluation ------------------

if __name__ == '__main__':
    # As before, using higher timeframe data first can help build a stable baseline.
    for instrument in config["instruments"]:
        print(f"\n=== Training DQN for instrument: {instrument['name']} ===")
        env_config = {
            "df": instrument["df"],
            "lot_size": instrument["lot_size"],
            "transaction_cost": instrument["transaction_cost"],
            "window_size": config["window_size"],
            "name": instrument["name"]
        }
        env = TradingEnv(env_config)
        check_env(env, warn=True)

        model = DQN("MultiInputPolicy", env, verbose=1, device=config["device"])
        model.learn(total_timesteps=10000)  # Adjust total_timesteps as needed

        obs = env.reset()
        done = False
        while not done:
            action, _states = model.predict(obs)
            obs, reward, done, info = env.step(action)

        metrics = env.get_metrics()
        print("\nTrade Log Details:")
        for trade in env.trade_log:
            if trade["exit_time"] is not None:
                points = (trade["exit_price"] - trade["entry_price"]) if trade["position"] == "long" else (trade["entry_price"] - trade["exit_price"])
                print(f"Entry: {trade['entry_time']}, Exit: {trade['exit_time']}, Position: {trade['position']}, "
                      f"Entry Price: {trade['entry_price']:.2f}, Exit Price: {trade['exit_price']:.2f}, "
                      f"Points: {points:.2f}, PnL: {trade['pnl']:.2f}, Reason: {trade['reason']}")
            else:
                print(f"Open trade at {trade['entry_time']}")

        print("\nSummary Metrics:")
        print(f"Initial Capital: {metrics.get('initial_capital'):.2f}")
        print(f"Final Capital: {metrics.get('final_capital'):.2f}")
        print(f"Total Trades: {metrics.get('total_trades')}")
        print(f"Win Rate: {metrics.get('win_rate'):.2f}%")
        print(f"Max Drawdown: {metrics.get('max_drawdown'):.2f}")
        print(f"Profit Factor: {metrics.get('profit_factor'):.2f}")